# 🏙️ Modul 1: Ekstraksi Data Kualitas Udara & Cuaca DKI Jakarta (2024)

**Tujuan Notebook:**
1. Mengambil data historis kualitas udara Jakarta (PM2.5, PM10, AQI, CO, NO2) secara per jam (*hourly*) sepanjang tahun 2024 dari **Open-Meteo Air Quality API**.
2. Mengambil data cuaca Jakarta (Suhu, Kelembaban, Curah Hujan, Kecepatan Angin) untuk periode yang sama dari **Open-Meteo Archive Weather API**.
3. Menggabungkan (*merge*) kedua dataset berdasarkan stempel waktu (*timestamp*).
4. Menyimpan dataset mentah ke folder `data/raw/`.

### 1. Import Library yang Dibutuhkan

In [1]:
import requests
import pandas as pd
import numpy as np
import os

# Pastikan folder data/raw dan data/processed tersedia
os.makedirs("../data/raw", exist_ok=True)
os.makedirs("../data/processed", exist_ok=True)

print("Pandas version:", pd.__version__)

C:\Users\ADNIN DANIL AZKIA\AppData\Roaming\Python\Python311\site-packages\pandas\core\computation\expressions.py:23: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.8.4' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED
C:\Users\ADNIN DANIL AZKIA\AppData\Roaming\Python\Python311\site-packages\pandas\core\arrays\masked.py:56: UserWarning: Pandas requires version '1.4.2' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (


Pandas version: 3.0.6


### 2. Parameter Lokasi & Periode Waktu
Kita menggunakan koordinat pusat DKI Jakarta:
- Latitude: `-6.2088`
- Longitude: `106.8456`
- Zona Waktu: `Asia/Jakarta` (UTC+7)
- Periode: 1 Januari 2024 - 31 Desember 2024 (1 tahun penuh = 8.784 jam)

In [2]:
LATITUDE = -6.2088
LONGITUDE = 106.8456
START_DATE = "2024-01-01"
END_DATE = "2024-12-31"
TIMEZONE = "Asia/Bangkok"  # UTC+7 setara WIB

### 3. Ekstraksi Data Kualitas Udara (Air Quality)

In [3]:
aq_url = "https://air-quality-api.open-meteo.com/v1/air-quality"
aq_params = {
    "latitude": LATITUDE,
    "longitude": LONGITUDE,
    "start_date": START_DATE,
    "end_date": END_DATE,
    "hourly": "pm10,pm2_5,carbon_monoxide,nitrogen_dioxide,sulphur_dioxide,us_aqi",
    "timezone": TIMEZONE
}

print("Mengunduh data kualitas udara Jakarta 2024...")
response_aq = requests.get(aq_url, params=aq_params)
data_aq = response_aq.json()

# Konversi ke pandas DataFrame
df_aq = pd.DataFrame(data_aq["hourly"])
df_aq["time"] = pd.to_datetime(df_aq["time"])
print(f"Data kualitas udara berhasil diambil: {df_aq.shape[0]} baris, {df_aq.shape[1]} kolom")
df_aq.head()

Mengunduh data kualitas udara Jakarta 2024...
Data kualitas udara berhasil diambil: 8784 baris, 7 kolom


,time,pm10,pm2_5,carbon_monoxide,nitrogen_dioxide,sulphur_dioxide,us_aqi
0,2024-01-01 00:00:00,135.0,94.5,3010.0,85.2,51.4,158
1,2024-01-01 01:00:00,128.1,89.6,3048.0,81.3,47.3,159
2,2024-01-01 02:00:00,120.1,84.1,2898.0,73.5,42.3,160
3,2024-01-01 03:00:00,106.9,74.7,2650.0,64.0,37.2,160
4,2024-01-01 04:00:00,98.5,68.9,2442.0,57.5,35.0,160


### 4. Ekstraksi Data Cuaca Historis (Weather Archive)

In [4]:
weather_url = "https://archive-api.open-meteo.com/v1/archive"
weather_params = {
    "latitude": LATITUDE,
    "longitude": LONGITUDE,
    "start_date": START_DATE,
    "end_date": END_DATE,
    "hourly": "temperature_2m,relative_humidity_2m,precipitation,rain,wind_speed_10m",
    "timezone": TIMEZONE
}

print("Mengunduh data cuaca Jakarta 2024...")
response_weather = requests.get(weather_url, params=weather_params)
data_weather = response_weather.json()

# Konversi ke pandas DataFrame
df_weather = pd.DataFrame(data_weather["hourly"])
df_weather["time"] = pd.to_datetime(df_weather["time"])
print(f"Data cuaca berhasil diambil: {df_weather.shape[0]} baris, {df_weather.shape[1]} kolom")
df_weather.head()

Mengunduh data cuaca Jakarta 2024...
Data cuaca berhasil diambil: 8784 baris, 6 kolom


,time,temperature_2m,relative_humidity_2m,precipitation,rain,wind_speed_10m
0,2024-01-01 00:00:00,25.1,97,0.0,0.0,2.2
1,2024-01-01 01:00:00,24.8,98,0.0,0.0,2.5
2,2024-01-01 02:00:00,24.4,99,0.0,0.0,1.8
3,2024-01-01 03:00:00,24.4,99,0.0,0.0,2.2
4,2024-01-01 04:00:00,24.3,99,0.0,0.0,3.9


### 5. Penggabungan Dataset (Merge on Timestamp)

In [5]:
df_combined = pd.merge(df_aq, df_weather, on="time", how="inner")

# Menambahkan fitur waktu turunan (feature engineering sederhana)
df_combined["date"] = df_combined["time"].dt.date
df_combined["hour"] = df_combined["time"].dt.hour
df_combined["day_name"] = df_combined["time"].dt.day_name()
df_combined["month"] = df_combined["time"].dt.month
df_combined["is_weekend"] = df_combined["time"].dt.dayofweek.isin([5, 6])

print("Ringkasan dataset gabungan:")
print(df_combined.info())
df_combined.head()

Ringkasan dataset gabungan:
<class 'pandas.DataFrame'>
RangeIndex: 8784 entries, 0 to 8783
Data columns (total 17 columns):
 #   Column                Non-Null Count  Dtype         
---  ------                --------------  -----         
 0   time                  8784 non-null   datetime64[us]
 1   pm10                  8784 non-null   float64       
 2   pm2_5                 8784 non-null   float64       
 3   carbon_monoxide       8784 non-null   float64       
 4   nitrogen_dioxide      8784 non-null   float64       
 5   sulphur_dioxide       8784 non-null   float64       
 6   us_aqi                8784 non-null   int64         
 7   temperature_2m        8784 non-null   float64       
 8   relative_humidity_2m  8784 non-null   int64         
 9   precipitation         8784 non-null   float64       
 10  rain                  8784 non-null   float64       
 11  wind_speed_10m        8784 non-null   float64       
 12  date                  8784 non-null   object        
 13  h

,time,pm10,pm2_5,carbon_monoxide,nitrogen_dioxide,sulphur_dioxide,us_aqi,temperature_2m,relative_humidity_2m,precipitation,rain,wind_speed_10m,date,hour,day_name,month,is_weekend
0,2024-01-01 00:00:00,135.0,94.5,3010.0,85.2,51.4,158,25.1,97,0.0,0.0,2.2,2024-01-01,0,Monday,1,False
1,2024-01-01 01:00:00,128.1,89.6,3048.0,81.3,47.3,159,24.8,98,0.0,0.0,2.5,2024-01-01,1,Monday,1,False
2,2024-01-01 02:00:00,120.1,84.1,2898.0,73.5,42.3,160,24.4,99,0.0,0.0,1.8,2024-01-01,2,Monday,1,False
3,2024-01-01 03:00:00,106.9,74.7,2650.0,64.0,37.2,160,24.4,99,0.0,0.0,2.2,2024-01-01,3,Monday,1,False
4,2024-01-01 04:00:00,98.5,68.9,2442.0,57.5,35.0,160,24.3,99,0.0,0.0,3.9,2024-01-01,4,Monday,1,False


### 6. Simpan Dataset Mentah ke Disk
Kita simpan file ini ke `data/raw/jakarta_environment_hourly_2024.csv`.

In [6]:
output_path = "../data/raw/jakarta_environment_hourly_2024.csv"
df_combined.to_csv(output_path, index=False)
print(f"Dataset berhasil disimpan ke: {output_path}")
print(f"Total baris: {len(df_combined):,}")

Dataset berhasil disimpan ke: ../data/raw/jakarta_environment_hourly_2024.csv
Total baris: 8,784
